# MLFlow example

## Start LMFlow server

### Locally

```bash
mlflow server \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns \
  --host 127.0.0.1 \
  --port 5005
```

### from docker-compose
```bash
mlflow-docker-compose.yml
```

## Logging experiment

In [1]:
import mlflow
from mlflow.models.signature import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import pickle

In [2]:
# Подключение к локальному MLflow серверу
mlflow.set_tracking_uri("http://127.0.0.1:5005")

# Создаем или подключаемся к эксперименту
experiment_name = "Iris_Classification"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1753695455196, experiment_id='1', last_update_time=1753695455196, lifecycle_stage='active', name='Iris_Classification', tags={}>

In [3]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [4]:
for C in [0.1, 1.0]:
    with mlflow.start_run(run_name=f"Run_C={C}") as run:
        model = LogisticRegression(C=C, max_iter=200)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="macro")

        mlflow.log_param("C", C)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)

        # Сохраняем модель как бинарный объект (pickle)
        with open("model.pkl", "wb") as f:
            pickle.dump(model, f)
        mlflow.log_artifact("model.pkl", artifact_path="model_pickle")
        

🏃 View run Run_C=0.1 at: http://127.0.0.1:5005/#/experiments/1/runs/24365cfdacff4a3c86643a85ee89f5ba
🧪 View experiment at: http://127.0.0.1:5005/#/experiments/1
🏃 View run Run_C=1.0 at: http://127.0.0.1:5005/#/experiments/1/runs/2e8221fbd5c2403ea4c64f40fe1db898
🧪 View experiment at: http://127.0.0.1:5005/#/experiments/1


##  Получение списка экспериментов и запусков

In [5]:
from mlflow.tracking import MlflowClient

In [6]:
mlflow.set_tracking_uri("http://127.0.0.1:5005")
client = MlflowClient()

In [7]:
# Получаем ID эксперимента
experiment = client.get_experiment_by_name("Iris_Classification")
experiment_id = experiment.experiment_id

In [8]:
# Получаем все запуски
runs = client.search_runs(experiment_id, order_by=["metrics.f1_score DESC"])

In [9]:
# Выбираем лучший запуск
best_run = runs[0]
print(f"Лучший запуск: {best_run.info.run_id}, f1_score = {best_run.data.metrics['f1_score']}")

Лучший запуск: 2e8221fbd5c2403ea4c64f40fe1db898, f1_score = 1.0


## Загрузка модели из лучшего запуска

In [10]:
from mlflow.artifacts import download_artifacts
import pickle

local_path = download_artifacts(
    run_id=best_run.info.run_id,
    artifact_path="model_pickle/model.pkl"
)

with open(local_path, "rb") as f:
    model = pickle.load(f)

print("Модель загружена. Пример предсказания:")
print(model.predict([X_test[0]]))

Модель загружена. Пример предсказания:
[1]


## Проблема переносимости

pickle (а также joblib) не подходит для долговременного и переносимого хранения моделей, особенно:
- при смене версий Python / библиотек (например, scikit-learn, PyTorch, XGBoost)
- при переносе между ОС и архитектурами (x86 ↔ ARM)
- при деплое в прод без контроля зависимостей

### Лучший вариант: MLflow + ONNX
1.	Конвертируй модель в ONNX (например, scikit-learn → ONNX)
2.	Логируй её в MLflow как ONNX flavor

In [29]:
import os
import numpy as np
import torch
import torch.nn as nn
import mlflow
import mlflow.onnx
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from mlflow.models.signature import infer_signature
from mlflow.artifacts import download_artifacts
import onnxruntime as ort
import onnx

In [22]:
# --------- Простая PyTorch модель ---------
class IrisNet(nn.Module):
    def __init__(self):
        super(IrisNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(4, 10),
            nn.ReLU(),
            nn.Linear(10, 3)
        )

    def forward(self, x):
        return self.model(x)

In [23]:
# --------- MLflow настройки ---------
mlflow.set_tracking_uri("http://127.0.0.1:5005")
mlflow.set_experiment("Iris_PyTorch_ONNX")

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1753697792128, experiment_id='2', last_update_time=1753697792128, lifecycle_stage='active', name='Iris_PyTorch_ONNX', tags={}>

In [30]:
# --------- Обучение и логирование ---------
for lr in [0.01, 0.05]:
    with mlflow.start_run(run_name=f"LR={lr}") as run:
        model = IrisNet()
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        X_tensor = torch.tensor(X_train, dtype=torch.float32)
        y_tensor = torch.tensor(y_train)

        for epoch in range(200):
            optimizer.zero_grad()
            output = model(X_tensor)
            loss = criterion(output, y_tensor)
            loss.backward()
            optimizer.step()

        # Оценка
        model.eval()
        with torch.no_grad():
            y_pred = model(torch.tensor(X_test, dtype=torch.float32)).argmax(dim=1).numpy()
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average="macro")

        # Логируем параметры и метрики
        mlflow.log_param("learning_rate", lr)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)

        # ONNX экспорт
        dummy_input = torch.randn(1, 4, dtype=torch.float32)
        onnx_path = f"model_{lr}.onnx"
        torch.onnx.export(
            model, dummy_input, onnx_path,
            export_params=True, opset_version=11,
            input_names=["input"], output_names=["output"],
            dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
        )

        # логируем модель
        onnx_model = onnx.load(onnx_path)

        mlflow.onnx.log_model(
            onnx_model=onnx_model,
            artifact_path="onnx_model",
            input_example=X_train[:2],
            signature=infer_signature(
                X_train,
                model(torch.tensor(X_train, dtype=torch.float32)).detach().numpy()
            )
        )

🏃 View run LR=0.01 at: http://127.0.0.1:5005/#/experiments/2/runs/9b3c7bb8f5f64af69260b73bf4baca2a
🧪 View experiment at: http://127.0.0.1:5005/#/experiments/2
🏃 View run LR=0.05 at: http://127.0.0.1:5005/#/experiments/2/runs/fe71fe52201242549b82be54e998a59c
🧪 View experiment at: http://127.0.0.1:5005/#/experiments/2


In [31]:
# --------- Загрузка лучшей модели ---------
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("Iris_PyTorch_ONNX")
runs = client.search_runs(experiment.experiment_id, order_by=["metrics.f1_score DESC"])
best_run = runs[0]
run_id = best_run.info.run_id

onnx_model_path = download_artifacts(run_id=run_id, artifact_path="onnx_model/model.onnx")

In [49]:
# --------- Инференс через ONNX ---------
session = ort.InferenceSession(onnx_model_path)
input_name = session.get_inputs()[0].name
x = X_test[0].reshape(1, -1).astype(np.float32)
pred = session.run(None, {input_name: x})[0]
predicted_class = np.argmax(pred, axis=1)

print("✅ Предсказанный класс:", predicted_class)

✅ Предсказанный класс: [1]


In [50]:
x

array([[6.1, 2.8, 4.7, 1.2]], dtype=float32)

## MLFlow model serve

### Получение run_id лучшего запуска по метрике f1_score

In [33]:
import mlflow
from mlflow.tracking import MlflowClient

In [39]:

mlflow.set_tracking_uri("http://127.0.0.1:5005")  # Укажи свой URI

client = MlflowClient()

# Получаем эксперимент по имени
experiment_name = "Iris_PyTorch_ONNX"
experiment = client.get_experiment_by_name(experiment_name)
experiment_id = experiment.experiment_id

# Получаем все запуски, отсортированные по f1_score по убыванию
runs = client.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.f1_score DESC"],
    max_results=1
)

# Получаем лучший run
best_run = runs[0]
run_id = best_run.info.run_id
print(f"Лучший run_id: {run_id}")

Лучший run_id: fe71fe52201242549b82be54e998a59c


### Запуск MLFlow serve server

```bash
export MLFLOW_TRACKING_URI=http://127.0.0.1:5005
```

```bash
mlflow models serve \
  -m "runs:/<RUN_ID>/onnx_model" \
  --no-conda \
  -h 0.0.0.0 \
  -p 5001
```

### Получаем результат работы модели

In [45]:
import requests

In [51]:
data = {
    # "inputs": [[5.1, 3.5, 1.4, 0.2]]
    "inputs": [[6.1, 2.8, 4.7, 1.2]]
}

response = requests.post("http://127.0.0.1:5001/invocations", json=data)

logits = response.json()["predictions"]["output"]
predicted_class = int(np.argmax(logits[0]))

print("Класс:", predicted_class)

Класс: 1
